# Spectral Clustering on Graph Data
**Assignment:** Perform spectral clustering on the provided graph, identify the number of clusters, and print the cluster number for each node.

## 1. Problem Understanding

Spectral clustering is a graph-based clustering technique that:
1. Builds an adjacency matrix from the graph
2. Computes the **Laplacian matrix** (L = D - A)
3. Finds the **eigenvalues and eigenvectors** of the Laplacian
4. Uses the **eigengap heuristic** to determine the number of clusters (k)
5. Applies **k-means** on the selected eigenvectors to assign cluster labels

**Dataset:** 600-node undirected graph with edge list as input.

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

## 3. Data Loading

In [ ]:
# Load the edge list
df = pd.read_csv('spectral_graph_600nodes_edges.csv')
print("Edge list (first 10 rows):")
print(df.head(10))
print(f"\nTotal edges: {len(df)}")
print(f"Columns: {list(df.columns)}")

## 4. Graph Construction

In [ ]:
# Build an undirected graph using NetworkX
G = nx.from_pandas_edgelist(df, source='Node 1', target='Node 2')

# Ensure all nodes 0..599 exist (add isolates if any)
G.add_nodes_from(range(600))

print(f"Number of nodes : {G.number_of_nodes()}")
print(f"Number of edges : {G.number_of_edges()}")
print(f"Is connected    : {nx.is_connected(G)}")
print(f"Number of connected components: {nx.number_connected_components(G)}")

In [ ]:
# Sort nodes so matrix indices are consistent: 0, 1, ..., 599
nodes = sorted(G.nodes())
n = len(nodes)
node_index = {node: i for i, node in enumerate(nodes)}

# Build the adjacency matrix (sparse)
rows, cols = [], []
for u, v in G.edges():
    i, j = node_index[u], node_index[v]
    rows += [i, j]
    cols += [j, i]

data = np.ones(len(rows))
A = csr_matrix((data, (rows, cols)), shape=(n, n))

# Degree matrix (diagonal)
degrees = np.array(A.sum(axis=1)).flatten()
D = csr_matrix((degrees, (range(n), range(n))), shape=(n, n))

# Unnormalized Laplacian: L = D - A
L = D - A

print(f"Adjacency matrix shape : {A.shape}")
print(f"Laplacian matrix shape : {L.shape}")
print(f"Min degree: {int(degrees.min())}  |  Max degree: {int(degrees.max())}  |  Mean degree: {degrees.mean():.2f}")

## 5. Spectral Clustering Steps

### Step 5a — Compute Eigenvalues and Eigenvectors

In [ ]:
# Compute the smallest 20 eigenvalues of the Laplacian
# (which=LM with sigma=0 gives smallest magnitude eigenvalues efficiently)
k_max = 20
eigenvalues, eigenvectors = eigsh(L, k=k_max, which='SM')

# Sort by eigenvalue (ascending)
idx = np.argsort(eigenvalues)
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f"Smallest {k_max} eigenvalues:")
for i, ev in enumerate(eigenvalues):
    print(f"  λ_{i:02d} = {ev:.6f}")

### Step 5b — Eigengap Plot (Determine k)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot eigenvalues
axes[0].plot(range(k_max), eigenvalues, 'o-', color='steelblue', markersize=6)
axes[0].set_xlabel('Index', fontsize=12)
axes[0].set_ylabel('Eigenvalue', fontsize=12)
axes[0].set_title('Smallest 20 Eigenvalues of the Laplacian', fontsize=13)
axes[0].grid(True, linestyle='--', alpha=0.6)

# Plot eigengaps
gaps = np.diff(eigenvalues)
axes[1].bar(range(1, k_max), gaps, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_xlabel('Index (gap between λ_i and λ_{i+1})', fontsize=12)
axes[1].set_ylabel('Eigengap', fontsize=12)
axes[1].set_title('Eigengap Heuristic', fontsize=13)
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('eigenvalue_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as eigenvalue_plot.png")

## 6. How the Number of Clusters Was Chosen

In [ ]:
# ── Eigengap heuristic ──────────────────────────────────────────────────────
# The number of clusters k equals the index AFTER the largest jump in
# consecutive eigenvalues. We skip λ_0 ≈ 0 (trivial) and look from index 1.

gaps = np.diff(eigenvalues)
best_gap_index = int(np.argmax(gaps[1:]) + 2)  # +2 because: +1 skip λ0, +1 for diff indexing
k = best_gap_index

print("Eigengaps (λ_{i+1} - λ_i):")
for i, g in enumerate(gaps[:15], start=1):
    marker = "  <── largest gap" if i == best_gap_index - 1 else ""
    print(f"  gap({i},{i+1}) = {g:.6f}{marker}")

print(f"\n✅ Number of clusters chosen by eigengap heuristic: k = {k}")

## 7. Apply Spectral Clustering

In [ ]:
# Select the first k eigenvectors (columns 0..k-1)
U = eigenvectors[:, :k]   # shape: (n, k)

# Row-normalize so each node vector has unit length (improves k-means stability)
U_norm = normalize(U, norm='l2', axis=1)

print(f"Embedding matrix shape (after selecting {k} eigenvectors): {U_norm.shape}")

In [ ]:
# Run k-means on the spectral embedding
np.random.seed(42)
kmeans = KMeans(n_clusters=k, n_init=20, max_iter=500, random_state=42)
kmeans.fit(U_norm)
cluster_labels = kmeans.labels_

print(f"K-Means converged in {kmeans.n_iter_} iterations.")
print(f"Inertia: {kmeans.inertia_:.4f}")

## 8. Final Cluster Label for Each Node

In [ ]:
# Build result DataFrame
result_df = pd.DataFrame({'Node': nodes, 'Cluster': cluster_labels})
result_df = result_df.sort_values('Node').reset_index(drop=True)

# Print all 600 nodes
print(f"{'Node':>6}  {'Cluster':>7}")
print("-" * 18)
for _, row in result_df.iterrows():
    print(f"{int(row['Node']):>6}  {int(row['Cluster']):>7}")

In [ ]:
# Summary statistics per cluster
print("\nCluster summary:")
summary = result_df.groupby('Cluster')['Node'].count().reset_index()
summary.columns = ['Cluster', 'Node Count']
print(summary.to_string(index=False))
print(f"\nTotal nodes: {len(result_df)}")

In [ ]:
cluster_sizes = result_df['Cluster'].value_counts().sort_index()

x = cluster_sizes.index.to_numpy()
y = cluster_sizes.to_numpy()

plt.figure(figsize=(8, 4))
plt.bar(x, y, color='steelblue', edgecolor='black', alpha=0.85)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Number of Nodes', fontsize=12)
plt.title(f'Node Count per Cluster (k={k})', fontsize=13)
plt.xticks(x)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('cluster_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved as cluster_sizes.png")

## 9. Save Output

In [ ]:
# Save cluster labels to CSV
result_df.to_csv('cluster_labels.csv', index=False)
print("✅ cluster_labels.csv saved successfully.")
print(result_df.head(10))

## 10. Conclusion

| Item | Value |
|------|-------|
| Total Nodes | 600 |
| Total Edges | 6734 |
| Method | Spectral Clustering (Unnormalized Laplacian + K-Means) |
| Cluster selection | Eigengap heuristic on smallest 20 eigenvalues |
| Number of clusters (k) | determined automatically from eigengap |

**Steps followed:**
1. Built the graph and computed the unnormalized Laplacian matrix `L = D - A`.
2. Computed the 20 smallest eigenvalues of `L` using sparse eigendecomposition.
3. Applied the **eigengap heuristic**: the largest jump in consecutive eigenvalues indicates the natural number of clusters.
4. Selected the first `k` eigenvectors as a low-dimensional embedding for each node.
5. Applied **row normalization** to the embedding and ran **K-Means** to assign cluster labels.
6. Saved all 600 node–cluster assignments to `cluster_labels.csv`.